# Discovering registered artists

plotlet's plot types live in a registry. Core artists register on `import plotlet`; 
extensions register lazily when you `import plotlet.extensions.<name>`; user-defined 
artists register via `pt.add_artist(...)`. This notebook shows `pt.artist_table()`, 
the helper for inspecting that registry.

In [14]:
import plotlet as pt


## Quick scan — `artist_table()`

The default view: name / origin / layer / coord_systems. 
Pretty-prints in the REPL and renders as an HTML table in Jupyter.

In [15]:
pt.artist_table()

name,origin,layer,coord_systems
annotate,core,foreground,-
area,core,data,{Circular}
axhline,core,foreground,-
axhspan,core,background,-
axvline,core,foreground,-
axvspan,core,background,-
bar,core,data,-
boxplot,core,data,-
contour,core,data,-
dendrogram,core,data,-


## All columns

Pass `columns="all"` to surface every informative `ArtistSpec` field. The column 
set is derived live from the dataclass, so new fields appear automatically without 
touching this helper. Required plumbing callables (`record`, `draw`) and lambda-defaulted 
fields (`xdomain`, `ydomain`) are skipped — they'd be `"fn"` on every row.

In [16]:
pt.artist_table(columns='all')

name,origin,layer,uses_color_cycle,default_color,accepts_data_positional,legend_entries,legend_gradient,data_attrs,flips_y_axis,tight_domain,force_zero_x,force_zero_y,axis_order,frame_defaults,crosses_sectors,coord_systems
annotate,core,foreground,False,#222222,False,-,-,fn,-,False,False,False,-,-,False,-
area,core,data,True,-,True,fn,-,fn,-,False,False,False,-,-,False,{Circular}
axhline,core,foreground,False,#000000,False,fn,-,fn,-,False,False,False,-,-,False,-
axhspan,core,background,False,#000000,True,fn,-,fn,-,False,False,False,-,-,False,-
axvline,core,foreground,False,#000000,False,fn,-,fn,-,False,False,False,-,-,False,-
axvspan,core,background,False,#000000,True,fn,-,fn,-,False,False,False,-,-,False,-
bar,core,data,True,-,True,fn,-,fn,-,False,fn,fn,-,-,False,-
boxplot,core,data,True,-,True,fn,-,-,-,False,False,False,-,-,False,-
contour,core,data,False,#1f77b4,False,-,fn,-,-,False,False,False,-,-,False,-
dendrogram,core,data,False,#1a1a1a,False,-,-,fn,-,True,False,False,fn,fn,True,-


## Custom column pick

Pass a list to choose columns explicitly, in the order you want them.

In [17]:
pt.artist_table(columns=['name', 'coord_systems', 'crosses_sectors'])

name,coord_systems,crosses_sectors
annotate,-,False
area,{Circular},False
axhline,-,False
axhspan,-,False
axvline,-,False
axvspan,-,False
bar,-,False
boxplot,-,False
contour,-,False
dendrogram,-,True


## Dynamic — extensions show up on import

Extensions don't auto-register; they're pay-as-you-go. Import one and it appears 
in the table on the next call, tagged `extension`.

In [18]:
import plotlet.extensions.volcano       # registers the volcano artist
import plotlet.extensions.manhattan     # ditto

[r for r in pt.artist_table() if r['origin'] == 'extension']

[{'origin': 'extension',
  'name': 'manhattan',
  'record': <function plotlet.extensions.manhattan.manhattan_record(args, kw)>,
  'draw': <function plotlet.extensions.manhattan.manhattan_draw(a, ctx)>,
  'xdomain': <function plotlet.extensions.manhattan.manhattan_xdomain(a)>,
  'ydomain': <function plotlet.extensions.manhattan.manhattan_ydomain(a)>,
  'layer': 'data',
  'uses_color_cycle': False,
  'default_color': None,
  'accepts_data_positional': True,
  'legend_entries': None,
  'legend_gradient': None,
  'data_attrs': None,
  'flips_y_axis': None,
  'tight_domain': False,
  'force_zero_x': False,
  'force_zero_y': False,
  'axis_order': None,
  'frame_defaults': None,
  'crosses_sectors': False,
  'coord_systems': set()},
 {'origin': 'extension',
  'name': 'volcano',
  'record': <function plotlet.extensions.volcano.volcano_record(args, kw)>,
  'draw': <function plotlet.extensions.volcano.volcano_draw(a, ctx)>,
  'xdomain': <function plotlet.extensions.volcano.volcano_xdomain(a)>,


## Dynamic — user-added artists

Anything registered via `pt.add_artist(...)` from a non-`plotlet.*` module is 
tagged `user`. Useful for sanity-checking your own extensions.

In [19]:
from plotlet.registry import ArtistSpec

pt.add_artist(ArtistSpec(
    name='my_artist',
    record=lambda args, kw: {'type': 'my_artist', 'opts': kw},
    draw=lambda a, ctx: '',
))

[r for r in pt.artist_table() if r['origin'] == 'user']

[{'origin': 'user',
  'name': 'my_artist',
  'record': <function __main__.<lambda>(args, kw)>,
  'draw': <function __main__.<lambda>(a, ctx)>,
  'xdomain': <function plotlet.registry.ArtistSpec.<lambda>(a)>,
  'ydomain': <function plotlet.registry.ArtistSpec.<lambda>(a)>,
  'layer': 'data',
  'uses_color_cycle': True,
  'default_color': None,
  'accepts_data_positional': True,
  'legend_entries': None,
  'legend_gradient': None,
  'data_attrs': None,
  'flips_y_axis': None,
  'tight_domain': False,
  'force_zero_x': False,
  'force_zero_y': False,
  'axis_order': None,
  'frame_defaults': None,
  'crosses_sectors': False,
  'coord_systems': set()}]

## Programmatic filtering

The table iterates as a list of dicts carrying the **full** field set 
(regardless of which columns are displayed). Use comprehensions to filter:

In [20]:
# all foreground-layer artists
[r['name'] for r in pt.artist_table() if r['layer'] == 'foreground']

['annotate', 'axhline', 'axvline', 'rug', 'text']

In [21]:
# all artists that draw correctly under non-affine coords (CircularCoordinate, ...)
[r['name'] for r in pt.artist_table() if r['coord_systems']]

['area', 'fill_between', 'heatmap', 'hist', 'line', 'scatter', 'step']